In [95]:
import pandas as pd
import numpy as np

In [96]:
data = pd.read_csv("autos.csv", delimiter=',' ,encoding='latin-1')

#cleaning data
data = data.drop(columns='postalCode')
data = data.drop(columns='nrOfPictures')
data = data[data.yearOfRegistration <= 2016]
data = data[data.yearOfRegistration >= 1995]

data['dateCreated'] = pd.to_datetime(data['dateCreated'])
data['lastSeen'] = pd.to_datetime(data['lastSeen'])
data['listingAge'] = (data['lastSeen'] - data['dateCreated']) / pd.Timedelta(days=1)
#data['logPrice'] = log1p(data['price])
data = data.drop(columns=['dateCreated', 'lastSeen'])

data = data.dropna(subset=['model', 'gearbox', 'fuelType'])

data = data[data['price'] > 500]
data = data[data['price'] < 100000]

data = data[data['powerPS'] < 1000]
data = data[data['powerPS'] > 5]

data = data[data.offerType == 'Angebot']
data = data.drop(columns = 'offerType')
data['car_age'] = 2016 - data['yearOfRegistration']
data = data.drop(columns = ['seller', 'name', 'dateCrawled', 'yearOfRegistration', 'abtest', 'monthOfRegistration']) 

In [97]:
#Splitting the data

from sklearn.model_selection import train_test_split

numeric_features = ['powerPS', 'kilometer', 'car_age', 'listingAge']
categorical_features = ['vehicleType', 'gearbox', 'model', 'fuelType', 'brand', 'notRepairedDamage', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

y = data['price'].copy()
X = data[categorical_features + numeric_features].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, random_state = 42, test_size = 0.2)





In [65]:
#Standard Random Forest Regressor
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OneHotEncoder(handle_unknown = "ignore", sparse_output = False, max_categories = 50)

seed = 42

def make_forest(criterion = "squared_error", max_depth = 3, min_samples_leaf = 5):
    prepare = ColumnTransformer([
        ("numeric", SimpleImputer(strategy = "mean"), numeric_features),
        ("categorical", ohe, categorical_features)
    ])
    return Pipeline([
        ("prepare", prepare),
        ("forest", RandomForestRegressor(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state = seed)),
         ])

#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_leaves": sum(t.get_n_leaves() for t in model.named_steps["forest"].estimators_) / len(model.named_steps["forest"].estimators_),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "criterion" : ["squared_error"], "max_depth" : [3, 6, 9], "min_samples_leaf" : [5, 10, 20]
})
depth_labels = [str(settings["max_depth"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 5} in 41.4s
[2/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 10} in 41.9s
[3/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 20} in 40.1s
[4/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 5} in 62.2s
[5/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 10} in 60.6s
[6/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 20} in 60.0s
[7/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 5} in 73.9s
[8/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 10} in 74.3s
[9/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 20} in 80.2s


,settings,training MAE,validation MAE,training RMSE,validation RMSE,val_r2,avg_leaves
0,"{'criterion': 'squared_error', 'max_depth': 3,...",2797.167,2788.479,4631.157,4602.542,0.642,8.00
1,"{'criterion': 'squared_error', 'max_depth': 3,...",2797.167,2788.479,4631.157,4602.542,0.642,8.00
2,"{'criterion': 'squared_error', 'max_depth': 3,...",2797.167,2788.479,4631.157,4602.542,0.642,8.00
3,"{'criterion': 'squared_error', 'max_depth': 6,...",1960.990,1944.468,3331.081,3307.777,0.815,62.65
4,"{'criterion': 'squared_error', 'max_depth': 6,...",1961.147,1944.301,3332.630,3308.202,0.815,61.23
5,"{'criterion': 'squared_error', 'max_depth': 6,...",1962.656,1944.017,3342.833,3302.809,0.816,61.81
6,"{'criterion': 'squared_error', 'max_depth': 9,...",1587.534,1599.518,2688.790,2759.215,0.871,426.61
7,"{'criterion': 'squared_error', 'max_depth': 9,...",1593.973,1601.195,2729.178,2776.169,0.870,388.94
8,"{'criterion': 'squared_error', 'max_depth': 9,...",1608.582,1611.028,2799.552,2813.801,0.866,346.31


In [98]:
#Standard HistGradientBoostRegressor
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OrdinalEncoder(handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-2,
    max_categories=254,)
cat_indices = list(range(len(numeric_features), len(numeric_features) + len(categorical_features)))

seed = 42

def make_forest(loss = "absolute_error", learning_rate = 0.1, max_iter= 300, max_leaf_nodes = 31, min_samples_leaf = 5, l2_regularization=0.0,):
    prepare = ColumnTransformer([
        ("numeric", "passthrough", numeric_features),
        ("categorical", ohe, categorical_features),
    ])
    pipe = Pipeline([
        ("prepare", prepare),
        ("forest", HistGradientBoostingRegressor(loss=loss,
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            categorical_features=cat_indices,
            early_stopping=True,      
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=seed,)),
         ])
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,
        inverse_func=np.expm1,)
        
#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)

        predictors = model.regressor_.named_steps["forest"]._predictors
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_nodes": sum(p[0].nodes.shape[0] for p in predictors) / len(predictors),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "loss" : ["absolute_error"], "learning_rate": [0.05], "max_iter" : [1000], "max_leaf_nodes" : [127, 255], "min_samples_leaf" : [15, 20, 30], "l2_regularization": [0.1],
})
leaf_labels = [str(settings["max_leaf_nodes"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 15} in 64.4s
[2/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 20} in 69.5s
[3/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 30} in 80.4s
[4/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 15} in 97.6s
[5/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 20} in 69.4s
[6/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 30} in 80.1s


,settings,training MAE,validation MAE,training RMSE,validation RMSE,val_r2,avg_nodes
0,"{'l2_regularization': 0.1, 'learning_rate': 0....",950.044,1080.021,1874.111,2127.231,0.924,253.0
1,"{'l2_regularization': 0.1, 'learning_rate': 0....",954.244,1081.133,1877.236,2134.300,0.923,253.0
2,"{'l2_regularization': 0.1, 'learning_rate': 0....",948.993,1080.688,1887.452,2137.619,0.923,253.0
3,"{'l2_regularization': 0.1, 'learning_rate': 0....",887.957,1071.519,1791.193,2117.813,0.924,509.0
4,"{'l2_regularization': 0.1, 'learning_rate': 0....",918.844,1078.094,1842.171,2140.682,0.923,509.0
5,"{'l2_regularization': 0.1, 'learning_rate': 0....",921.640,1077.158,1855.246,2145.108,0.922,509.0
